# Phase 1 — Comprehensive Data Quality & Profiling Dashboard

This interactive notebook executes the complete Data Quality Framework across the operational PostgreSQL database (`healthcare_dev`). It provides deep visibility into table volumes, column missingness, business key duplicates, foreign key referential integrity, clinical vital validations, and multidimensional quality scoring.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Set project root in path
sys.path.insert(0, str(Path.cwd().parent))

from src.profiling.row_counts import profile_row_counts
from src.profiling.null_analysis import analyze_nulls
from src.profiling.duplicates import detect_duplicates
from src.profiling.integrity import check_referential_integrity
from src.profiling.temporal import check_temporal_integrity
from src.profiling.validation import get_validation_summary
from src.profiling.outliers import detect_outliers
from src.profiling.quality_score import calculate_quality_scores

print("Data Quality Engine Loaded Successfully.")

## 1. Table Inventory & Operational Row Counts

Profiling operational data volumes, schema width, primary keys, and temporal event bounds.

In [ ]:
df_tables = profile_row_counts()
print(f"Total Operational Tables Profiled: {len(df_tables)}")
display(df_tables.head(15))

## 2. Missing Value Analysis & Sparsity Profiling

Assessing completeness per column, isolating mandatory violations from expected demographic optionality.

In [ ]:
df_nulls = analyze_nulls()
print(f"Total Evaluated Columns: {len(df_nulls)}")
# Show columns with highest null rates
display(df_nulls[df_nulls["Null_Percentage"] > 0].sort_values(by="Null_Percentage", ascending=False).head(15))

## 3. Duplicate Detection & Identity Overlaps

Scanning for shared phone numbers, email collisions, double-booked appointments, and duplicate prescription line items.

In [ ]:
df_dups = detect_duplicates()
display(df_dups)

## 4. Referential Integrity (Zero-Orphan Verification)

Executing anti-join queries across all clinical parent-child relationships.

In [ ]:
df_integrity = check_referential_integrity()
total_orphans = df_integrity["Invalid_Orphan_Records"].sum()
print(f"Total Orphan Records Detected: {total_orphans}")
display(df_integrity)

## 5. Clinical Physiological Range Validations

Testing vitals against medical plausibility boundaries (Blood pressure, Heart rate, SpO2, Body temperature, Respiratory rate).

In [ ]:
df_valid = get_validation_summary()
display(df_valid)

## 6. Statistical Outlier Detection

Evaluating distributions of OPD waiting times, teleconsultation durations, and booking lead times using IQR and Z-scores.

In [ ]:
df_outliers = detect_outliers()
display(df_outliers)

## 7. Multidimensional Data Quality Scorecard

Synthesizing Completeness (25%), Consistency (25%), Validity (20%), Integrity (20%), and Timeliness (10%).

In [ ]:
score_results = calculate_quality_scores()
print(f"Overall Platform Quality Score: {score_results['platform_score']} / 100")
display(score_results["table_scores_df"])